***

**OUR APPROACH:**

After exploratory analysis and testing, we discovered an important issue with the MusicBrainz schema: directly linking artists to genres through the l_artist_genre linking table was not practical because the table is almost empty. Joining through this relationship produced a dataset of only about 400 rows, which was too small for meaningful model training.

We leveraged the richer tagging system in MusicBrainz to adress this issue. Tags and genres already have substantial semantic overlap (tags such as “rock”, “jazz”, or “metal” often function as genre labels), and using tags dramatically increased the amount of usable data. This alternative approach produced a dataset of approximately 1.5 million rows.

Using this strategy, we constructed a table containing:

- release_name
- artist_name
- area_name
- label_name
- tag_name (a list of tags associated with the release)

The first tag in the tag list was selected and treated as the target variable genre_name (since the first tag is often the most representative label for the release). The remaining tags were preserved as additional contextual features and stored as a vector under tags.

The model predicts genre_name (derived from the first tag) using features such as:

- artist name
- geographic area
- record label
- associated tags

while release_name primarily serves as a relational anchor connecting the metadata together rather than as a predictive feature itself.

Additional preprocessing and feature engineering steps were then applied:

- missing-value imputation
- filtering infrequent categories
- reducing high-cardinality categorical variables (area_name, tags, and genre labels)

These transformations reduced the problem to a 19 genre classification task.

We used several Spark ML preprocessing components:

- CountVectorizer to convert tag lists into sparse binary vectors
- StringIndexer to encode categorical variables such as genre, artist area, and label
- VectorAssembler to combine all engineered features into a single feature vector suitable for distributed machine learning models

This final dataset and preprocessing pipeline were then used to train distributed classification models in Spark ML, including Random Forest and Decision Tree classifiers.

In [23]:
# pull in needed tables
release_df = dfs["release"].select(
    F.col("id").alias("release_id"),
    F.col("name").alias("release_name"),
    F.col("artist_credit").alias("release_artist_credit"),
    F.col("release_group").alias("release_release_group"),
)

acn_df = dfs["artist_credit_name"].select(
    F.col("artist_credit").alias("acn_artist_credit"),
    F.col("artist").alias("acn_artist_id"),
)

artist_df = dfs["artist"].select(
    F.col("id").alias("artist_id"),
    F.col("area").alias("artist_area_id"),
    F.col("gender").alias("artist_gender_id"),
)

gender_df = dfs["gender"].select(
    F.col("id").alias("gender_id"),
    F.col("name").alias("artist_gender"),
)

area_df = dfs["area"].select(
    F.col("id").alias("area_id"),
    F.col("name").alias("area_name"),
)

rgt_df = dfs["release_group_tag"].select(
    F.col("release_group").alias("rgt_release_group"),
    F.col("tag").alias("rgt_tag_id"),
    F.col("count").alias("tag_count"),
)

tag_df = dfs["tag"].select(
    F.col("id").alias("tag_id"),
    F.col("name").alias("tag_name"),
)

In [24]:
# build table (release name, artist gender, area name, tag name, tag count)
model_df = (
    release_df
    .join(acn_df, release_df["release_artist_credit"] == acn_df["acn_artist_credit"], "inner")
    .join(artist_df, acn_df["acn_artist_id"] == artist_df["artist_id"], "inner")
    .join(gender_df, artist_df["artist_gender_id"] == gender_df["gender_id"], "left")
    .join(area_df, artist_df["artist_area_id"] == area_df["area_id"], "left")
    .join(rgt_df, release_df["release_release_group"] == rgt_df["rgt_release_group"], "left")
    .join(tag_df, rgt_df["rgt_tag_id"] == tag_df["tag_id"], "left")
    .select("release_name", "artist_gender", "area_name", "tag_name", "tag_count")
)

In [25]:
# ranking tags
tag_window = Window.partitionBy("release_name").orderBy(F.col("tag_count").desc_nulls_last())
ranked_df = model_df.withColumn("tag_rank", F.row_number().over(tag_window))

# aggregating on release_name, assigning top tag to "genre_name", storing the rest in "tags"
aggregated_df = (
    ranked_df
    .groupBy("release_name")
    .agg(
        F.first("artist_gender", ignorenulls=True).alias("artist_gender"),
        F.first("area_name", ignorenulls=True).alias("area_name"),
        F.max(F.when(F.col("tag_rank") == 1, F.col("tag_name"))).alias("genre_name"),
        F.collect_set(F.when(F.col("tag_rank") > 1, F.col("tag_name"))).alias("tags"),
    ).filter(F.col("genre_name").isNotNull())
)

# filter out garbage tags and genre names (using helper real_music_genres.csv file)
valid_genres_pd = pd.read_csv("real_music_genres.csv")
valid_genres = set(valid_genres_pd["Genre"].dropna().str.strip().str.lower())
aggregated_df = aggregated_df.filter(F.lower(F.col("genre_name")).isin(list(valid_genres)))
valid_genres_bc = spark.sparkContext.broadcast(valid_genres)
filter_tags_udf = F.udf(
    lambda tags: [t for t in (tags or []) if t and t.lower() in valid_genres_bc.value], ArrayType(StringType())
)
aggregated_df = aggregated_df.withColumn("tags", filter_tags_udf(F.col("tags")))

# map each genre_name to one of 19 broad buckets (using helper genre_buckets_wide.csv file)
buckets_pd = pd.read_csv("genre_buckets_wide.csv")
genre_to_bucket = {}
for bucket in buckets_pd.columns:
    for genre in buckets_pd[bucket].dropna():
        genre_clean = genre.strip().lower()
        if genre_clean:
            genre_to_bucket[genre_clean] = bucket

bucket_lookup_df = spark.createDataFrame([(k, v) for k, v in genre_to_bucket.items()], ["genre_key", "bucket_name"])
aggregated_df = (
    aggregated_df
    .join(F.broadcast(bucket_lookup_df),
          F.lower(F.col("genre_name")) == F.col("genre_key"), "left")
    .drop("genre_name", "genre_key")
    .withColumnRenamed("bucket_name", "genre_name")
    .filter(F.col("genre_name").isNotNull())
)

# randomly impute 75% of missing values in artist_gender as 'Male', rest as 'Female'
aggregated_df = aggregated_df.withColumn(
    "artist_gender",
    F.when(F.col("artist_gender").isNull(),
        F.when(F.rand(seed=42) < 0.75, F.lit("Male")).otherwise(F.lit("Female"))
    ).otherwise(F.col("artist_gender"))
)

# keep only the top 120 most common areas and drop everything else
top_areas = (aggregated_df.groupBy("area_name").count().orderBy(F.col("count").desc())
    .limit(120).select("area_name").rdd.flatMap(lambda r: [r[0]]).collect())
aggregated_df = aggregated_df.filter(F.col("area_name").isin(top_areas))

**Final dataframe BEFORE encoding:**

In [26]:
aggregated_df.printSchema()
aggregated_df = aggregated_df.cache()
print("Row count:", aggregated_df.count())
peek(aggregated_df)

root
 |-- release_name: string (nullable = true)
 |-- artist_gender: string (nullable = true)
 |-- area_name: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genre_name: string (nullable = true)

Row count: 938482


,release_name,artist_gender,area_name,tags,genre_name
0,!!! (The Lost Album),Male,United States,"[punk, rock]",Rock
1,!!!More!!! (theme from Mondo Cane),Male,United States,"[jazz, rock]",Rock
2,!Bailando!,Female,United States,[],Latin
3,!Franchesckaar!,Female,England,"[electro, house, breakbeat, pop, electronic, i...",Electronic / Dance
4,!Ich kann,Male,Germany,[],Pop
5,!K7 Flavour: Princess Superstar Mix,Female,New York,"[hip hop, rock]",Electronic / Dance
6,!Слушай,Male,Russia,"[electronic, euro house]",Electronic / Dance
7,"""...zwei Gefühle..."", Musik mit Leonardo / Not...",Male,Austria,[classical],Classical / Orchestral / Opera
8,"""180""",Female,Germany,[punk],Rock
9,"""7""",Male,United States,"[rock, rock and roll]",Rock


**Final dataframe AFTER encoding:**

In [27]:
# using CountVectorizer to convert tag list into a sparse binary vector
cv = CountVectorizer(inputCol="tags", outputCol="tag_vector", minDF=2.0)
cv_model = cv.fit(aggregated_df)
vectorized_df = cv_model.transform(aggregated_df)

# using StringIndexer to convert genre/gender/area into a numeric index
genre_indexer = StringIndexer(inputCol="genre_name", outputCol="genre_index", handleInvalid="keep")
gender_indexer = StringIndexer(inputCol="artist_gender", outputCol="gender_index", handleInvalid="keep")
area_indexer = StringIndexer(inputCol="area_name", outputCol="area_index", handleInvalid="keep")

# putting all features into a single "features" vector using VectorAssembler
assembler = VectorAssembler(inputCols=["gender_index", "area_index", "tag_vector"], outputCol="features", handleInvalid="keep")

# putting it all together
pipeline = Pipeline(stages=[genre_indexer, gender_indexer, area_indexer, assembler])
pipeline_model = pipeline.fit(vectorized_df)
final_df = pipeline_model.transform(vectorized_df)
final_df = final_df.select("release_name", "genre_index", "features").cache()

print("Tag vocabulary size:", len(cv_model.vocabulary))
genre_list = sorted([row[0] for row in aggregated_df.select("genre_name").distinct().collect()])
print("Genre count:", len(genre_list))
print("Genres:", genre_list)

print("Row count:", final_df.count())
peek(final_df)

Tag vocabulary size: 1118
Genre count: 19
Genres: ['Ambient / New Age', 'Blues', 'Classical / Orchestral / Opera', 'Country / Americana / Bluegrass', 'Electronic / Dance', 'Experimental / Avant-garde / Noise', 'Folk / Singer-Songwriter', 'Gospel / Christian / Spiritual', 'Hip-Hop / Rap', 'Jazz', 'Latin', 'Metal', 'Other', 'Pop', 'Punk / Hardcore / Emo', 'R&B / Soul / Funk', 'Reggae / Ska / Dub', 'Rock', 'World / Global']
Row count: 938482


,release_name,genre_index,features
0,!!! (The Lost Album),0.0,"(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,!!!More!!! (theme from Mondo Cane),0.0,"(0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."
2,!Bailando!,14.0,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,!Franchesckaar!,1.0,"(1.0, 32.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0,..."
4,!Ich kann,2.0,"(0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
5,!K7 Flavour: Princess Superstar Mix,1.0,"(1.0, 23.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
6,!Слушай,1.0,"(0.0, 11.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
7,"""...zwei Gefühle..."", Musik mit Leonardo / Not...",4.0,"(0.0, 12.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,..."
8,"""180""",0.0,"(1.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9,"""7""",0.0,"(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
